# Benchmark sin ground-truth — detector × tracker

## ¿Para qué sirve este notebook?

Sirve para **decidir, con datos, qué configuración del pipeline es la mejor** sin
tener anotaciones manuales (ground-truth). La anotación del equipo en Roboflow sigue
pausada, así que en lugar de medir *exactitud vs. humano* (mAP, MOTA, mIoU — que
necesitan GT), medimos **métricas sin referencia**: consistencia física/temporal de
las predicciones y costo computacional.

El resultado es una **tabla comparativa** de las configuraciones, con tres ejes:

1. **Eficiencia** — FPS y VRAM pico (el "filtro de la realidad": descarta lo inviable).
2. **Trayectoria** — qué tan lógicas/estables son los tracks (longitud, fragmentación,
   suavidad).
3. **Máscara** — qué tan estable es la segmentación en el tiempo (IoU temporal, jitter).

## ¿Por qué es un benchmark honesto (sin fuga de datos)?

El detector YOLO (`best.pt`) se entrenó en *fase_1* con auto-labels SAM3 de los **103
videos NO-testing** (splits 0+1, split por video, anti-leak). Los **20 videos de
testing (`split=2`) quedaron intocados** → YOLO nunca los vio. Por eso corremos sobre
el testing: es justo para ambos detectores (ni SAM3-only ni YOLO→SAM3 tienen ventaja
de memorización).

## ¿Qué se ejecuta?

**5 configuraciones reales** sobre **5 videos del testing** elegidos al azar con
semilla fija (reproducible). Cada video se corre **completo** (duran < 2 min).

## Requisitos (correr EN EL POD, con GPU)

- Código al día (`git pull` en `develop`); si `import src.eval` falla, `pip install -e .`.
- `.env` con `CONFIG_FILENAME=01_yolo_sam3_config.json` (trae las secciones `yolo` y
  `botsort`).
- Pesos YOLO en `assets/yolo/best.pt` (los usan las configs `yolo_sam3`).
- Modelo SAM3 en `assets/sam3` y los 5 videos seeded bajo `data/raw`.

Con eso, basta **ejecutar las celdas de arriba a abajo**.

## Paso 1 — Definir las 5 configuraciones y seleccionar los videos

### Las configuraciones (qué combinaciones existen de verdad)

El pipeline tiene dos motores: `mode="segmentation"` (solo SAM3-texto, sin tracking) y
`mode="tracking"` (detector inyectable × tracker inyectable, con `obj_id` estable). Las
combinaciones **reales** son:

| Detección/segmentación | Sin tracking | ByteTrack | BoT-SORT |
|---|---|---|---|
| **SAM3 (texto)** `sam3_text` | ✅ (baseline) | ✅ | ✅ |
| **YOLO→SAM3** `yolo_sam3` | ❌ no existe | ✅ | ✅ |

- **`sam3_text`**: SAM3 segmenta directo por *prompt de texto* ("robot", "orange ball").
- **`yolo_sam3`**: YOLO da cajas rápidas → SAM3 segmenta por *box-prompt* dentro de
  ellas (YOLO localiza, SAM3 segmenta).
- **ByteTrack**: tracker ligero (Kalman + IoU). **BoT-SORT**: añade compensación de
  movimiento de cámara (GMC), más robusto pero más pesado.

**¿Por qué solo 5 y no 6?** En `mode="segmentation"` el `detector` **se ignora** (ese
modo siempre usa SAM3-texto; YOLO no está cableado ahí). Por eso "YOLO→SAM3 sin
tracking" no existe: sería idéntico al baseline SAM3-texto. Se omite para no gastar una
corrida en una fila duplicada.

### Los videos

`benchmark_videos(n=5, seed=42)` lee `db_metadata.csv`, filtra el `split==2` (testing) y
muestrea 5 videos con semilla fija → **misma lista en cada corrida** (comparaciones
reproducibles entre configs y entre días).

In [ ]:
from src.core.batch import run_batch
from src.eval.benchmark import (
    aggregate_config,
    benchmark_videos,
    comparison_table,
)

# seed=36: el 42 incluía un video de ~13 min (~23k frames) que disparaba OOM.
videos = benchmark_videos(n=5, seed=36)
print("videos del benchmark (split=2, seed=36):")
for v in videos:
    print(" ", v)

# (label, mode, detector, tracker). tracker=None solo en el baseline sin tracking.
# Nota: en segmentation el detector se ignora -> el baseline es SAM3-texto puro.
CONFIGS = [
    ("sam3_text+none",      "segmentation", "sam3_text", None),
    ("sam3_text+bytetrack", "tracking",     "sam3_text", "bytetrack"),
    ("sam3_text+botsort",   "tracking",     "sam3_text", "botsort"),
    ("yolo_sam3+bytetrack", "tracking",     "yolo_sam3", "bytetrack"),
    ("yolo_sam3+botsort",   "tracking",     "yolo_sam3", "botsort"),
]
print(f"\n{len(CONFIGS)} configuraciones a evaluar.")

## Paso 2 — Correr cada configuración y medir (resiliente a OOM)

### Cómo lo hace

Por cada config se llama a `run_batch`, que corre la inferencia de los videos (SAM3 y
YOLO son **singletons cacheados**, se cargan una sola vez en todo el notebook) y
devuelve un **resumen por video** con el **timing** instrumentado (`fps`,
`peak_vram_mb`) y la ruta del JSON. Luego `aggregate_config` lee esos JSON **frescos** y
produce **una fila por config**.

### Parámetros clave

- `include_masks` — RLE en el JSON, necesario para las métricas de máscara. **Solo en
  tracking** (`incl_masks = mode == "tracking"`): en el baseline de segmentación el
  `obj_id` es inestable y un IoU temporal ahí sería espurio, así que corre **sin**
  máscara y sus columnas `mask_iou`/`com_jitter` quedan N/A.
- `MAX_FRAMES` — **tope de frames en tracking** para evitar **OOM de RAM**. En tracking
  con máscara, `track_video` acumula todos los frame-records con RLE de un video y arma
  un JSON enorme; un video largo (p. ej. 6 600 frames a ~60 fps) revienta el proceso del
  kernel **aunque sobre VRAM** (el cuello es la RAM de CPU, no la GPU). Con el cap se usa
  un **prefijo contiguo** de `MAX_FRAMES` (`sampling="auto"`); los videos cortos corren
  enteros. `None` = sin tope. El baseline (segmentación, sin máscara) es ligero y corre
  completo.
- `sampling` — `"auto"` en tracking (respeta `MAX_FRAMES`), `"all"` en el baseline.
- `render_video=False`, `overwrite=True` — solo el dato; reprocesa en cada config.

### Resiliencia (persistencia incremental + reanudación)

Cada fila se **escribe al CSV apenas se calcula** (`append`). Si el kernel muere a mitad,
**no se pierde lo avanzado**: pon `RESUME=True` y re-ejecuta — las configs ya guardadas
se **saltan** y solo se corren las que falten. Entre configs se libera memoria
(`gc.collect()` + `torch.cuda.empty_cache()`).

> Importante: con `RESUME=False` (default) el CSV se **borra al inicio** para empezar
> limpio (evita mezclar con una corrida de otro seed). Si cambiaste el seed, déjalo en
> `False`. Si solo quieres continuar un run truncado, ponlo en `True`.

### Por qué "una config a la vez"

Todas las configs escriben en la misma ruta `outputs/inference/<stem>/<stem>.json`. El
bucle **corre una config y la agrega de inmediato** (con el JSON fresco) antes de que la
siguiente lo sobrescriba; por eso cada fila se mide sobre **sus propios datos**.

In [ ]:
import gc

import pandas as pd
import torch

from src.utils import PROJECT_ROOT

CSV_PATH = PROJECT_ROOT / "outputs" / "benchmark" / "comparison.csv"
MAX_FRAMES = 2500  # tope de frames en TRACKING para evitar OOM de RAM en videos largos.
#                    None = video completo. Bájalo si el kernel sigue muriendo.
RESUME = False  # True para REANUDAR tras un crash: salta las configs ya guardadas.

# Arranque limpio salvo que se pida reanudar (evita mezclar con un CSV de otro seed).
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
if not RESUME and CSV_PATH.exists():
    CSV_PATH.unlink()
done_labels = set(pd.read_csv(CSV_PATH)["config"]) if CSV_PATH.exists() else set()
if done_labels:
    print("reanudando; configs ya guardadas:", done_labels)

for label, mode, detector, tracker in CONFIGS:
    if label in done_labels:
        print(f"skip {label} (ya en el CSV)")
        continue
    print(f"\n===== {label}  ({mode}) =====")
    incl_masks = mode == "tracking"  # máscara solo con obj_id estable
    # Cap de frames solo en tracking (segmentación es ligera: sin máscara).
    if mode == "tracking" and MAX_FRAMES is not None:
        smp, mf = "auto", MAX_FRAMES  # prefijo contiguo acotado
    else:
        smp, mf = "all", None  # video completo

    summary = run_batch(
        mode=mode,
        videos=videos,
        detector=detector,
        tracker=tracker,
        sampling=smp,
        max_frames=mf,
        include_masks=incl_masks,
        render_video=False,
        overwrite=True,
    )
    row = aggregate_config(label, summary)

    # Persistir la fila YA (append): si el kernel muere luego, no se pierde lo avanzado.
    comparison_table([row]).to_csv(
        CSV_PATH, mode="a", header=not CSV_PATH.exists(), index=False
    )
    print("fila guardada:", row)

    # Liberar memoria entre configs (RAM + caché de VRAM).
    del summary
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Paso 3 — Tabla comparativa

`comparison_table` arma el DataFrame (una fila por config, columnas ordenadas) y
`write_table` lo persiste como CSV en `outputs/benchmark/comparison.csv` (git-ignored).
El CSV queda como evidencia versionable del experimento.

In [ ]:
# La tabla se arma desde el CSV (que se fue escribiendo fila a fila en el Paso 2).
# Así sobrevive a un reinicio de kernel: basta poner RESUME=True y re-ejecutar para
# completar las configs que falten.
import pandas as pd

df = pd.read_csv(CSV_PATH)
print(f"tabla: {CSV_PATH}  ({len(df)}/{len(CONFIGS)} configs)")
df

## Cómo leer la tabla

| Columna | Qué mide | Mejor cuando |
|---|---|---|
| `fps` | throughput (frames/seg) | **mayor** (más rápido) |
| `peak_vram_mb` | VRAM pico del proceso | **menor** (menos memoria) |
| `tracklet_len` | frames promedio que vive un `obj_id` | **mayor** (menos fragmentación) |
| `frag_rate` | proxy de cambios de ID (un ID muere y otro nace cerca y justo después) | **menor** |
| `smoothness` | varianza de la aceleración del centroide | **menor** (movimiento más físico) |
| `mask_iou` | IoU de la máscara del mismo objeto entre frames consecutivos | **mayor** (máscara estable) |
| `com_jitter` | temblor del centro de masa de la máscara entre frames | **menor** |

### Advertencias de interpretación (importante)

- Son métricas **sin ground-truth**: miden **consistencia y eficiencia**, NO exactitud
  vs. humano. Un `frag_rate` bajo no garantiza que el ID sea *correcto* (podría estar
  fusionando dos robots distintos). Úsalas como evidencia comparativa, no como verdad.
- `tracklet_len`/`frag_rate`/`smoothness` no interpolan frames faltantes; son un proxy
  válido **comparando configs sobre los mismos videos** (que es justo lo que hacemos).
- La **decisión final es humana**: la tabla informa el balance eficiencia↔consistencia,
  no dictamina un "ganador" automático.

### Hipótesis de ingeniería a contrastar

Se espera que **YOLO→SAM3** sea más rápido (YOLO filtra el espacio antes de SAM3) y que
**BoT-SORT** reduzca fragmentación frente a ByteTrack en cámara en movimiento, a costa
de más cómputo. El "sweet spot" típico suele ser **YOLO→SAM3 + ByteTrack**. La tabla
confirma o refuta esto con datos de nuestros videos.